# Lesson 1: Project Architecture & RESTful API Design
### VisionStream — Scalable Backend for Assistive Tech

---

## Learning Objectives

By the end of this lesson, you will be able to:

1. **Explain** the difference between a single-device prototype and a scalable cloud backend
2. **Design** RESTful API endpoints that communicate with IoT hardware
3. **Write** Pydantic schemas that validate incoming sensor data
4. **Initialize** a production-grade Python project with proper structure
5. **Test** your API endpoints using FastAPI's built-in Swagger UI

---

## Context: From Prototype to Platform

Your helmet prototype was a standalone embedded system: one helmet, one user, one device.

VisionStream is the upgrade: a **cloud platform** that supports thousands of helmets simultaneously.

Think of it this way:

| | Prototype | VisionStream |
|---|---|---|
| Scale | 1 device | 10,000+ devices |
| Data storage | On-device (memory) | PostgreSQL cloud database |
| Access | Local only | Global, via HTTPS |
| Reliability | Single point of failure | Redundant, load-balanced |
| Analytics | None | Real-time dashboard, alert heatmaps |

## Section 1: System Architecture

Every time a helmet detects an obstacle, here is what happens:

```
┌──────────────────┐     POST /telemetry/upload      ┌─────────────────────┐
│  Helmet Device   │ ─────────────────────────────►  │  VisionStream API   │
│  (IoT client)    │ ◄─────────────────────────────  │  (FastAPI)          │
│                  │     201 Created                  └──────────┬──────────┘
│  Sends JSON:     │                                             │
│  - device_id     │                               ┌────────────┴──────────┐
│  - timestamp     │                               │                       │
│  - gps_lat/lon   │                       ┌───────▼──────┐    ┌──────────▼──────┐
│  - obstacle dist │                       │ Redis Cache  │    │  PostgreSQL DB  │
│  - alert_type    │                       │ (fast reads) │    │ (durable store) │
└──────────────────┘                       └──────────────┘    └─────────────────┘
```

### Three-Layer Architecture

In production backend systems, code is organized into layers so each part has a single responsibility:

| Layer | Files | Responsibility |
|-------|-------|----------------|
| **Router** | `app/routers/*.py` | Receive HTTP request, validate input, return response |
| **Service** | `app/services/*.py` | Business logic — talk to database, apply rules |
| **Model** | `app/models/*.py` | Define database table structure |
| **Schema** | `app/schemas/*.py` | Define API request/response shapes (Pydantic) |

> **Key Rule:** Routers should be *thin* — they call services and return results.
> All database logic lives in services.

## Section 2: RESTful API Fundamentals

REST (Representational State Transfer) is the standard design style for web APIs.
A REST API expresses operations as **HTTP methods on resource URLs**.

### HTTP Methods

| Method | Meaning | Use Case |
|--------|---------|----------|
| `POST` | Create | Register a new device, upload telemetry |
| `GET` | Read | Retrieve device info, query alert history |
| `PUT` | Replace | Update entire device record |
| `PATCH` | Partial update | Update only firmware_version |
| `DELETE` | Remove | Deregister a device |

### HTTP Status Codes (the ones you will use)

| Code | Name | When to use |
|------|------|-------------|
| `200 OK` | Success | Successful GET |
| `201 Created` | Created | Successful POST (new resource created) |
| `400 Bad Request` | Client error | Malformed request |
| `404 Not Found` | Not found | Device_id does not exist |
| `409 Conflict` | Duplicate | Device with that ID already registered |
| `422 Unprocessable Entity` | Validation failed | Pydantic rejected the payload |
| `500 Internal Server Error` | Server crash | Unexpected exception in your code |

### URL Design Rules

```
# ✅ Good REST URLs — use nouns, not verbs
POST   /devices/register          → Create a device
GET    /devices/{device_id}       → Get one device
POST   /telemetry/upload          → Upload a reading
GET    /alerts/history            → Query alerts

# ❌ Bad URLs — avoid verbs in paths
POST   /createDevice
GET    /getDeviceById?id=123
POST   /uploadTelemetryData
```

## Section 3: Designing the Telemetry Payload

Every helmet sends data in this JSON format:

```json
{
  "device_id":            "helmet-001-berkeley",
  "timestamp":            "2025-09-15T14:23:01Z",
  "gps_lat":              37.8719,
  "gps_lon":              -122.2585,
  "obstacle_distance_cm": 45,
  "alert_type":           "OBSTACLE_WARNING",
  "battery_level":        82
}
```

### Why validation matters

Without validation, a single buggy helmet could send:
- `gps_lat: 9999.0` — breaks map visualizations
- `battery_level: -5` — corrupts analytics
- `device_id: null` — crashes the database insert

**Pydantic** handles all of this automatically using `Field()` constraints.
If validation fails, FastAPI returns `422 Unprocessable Entity` — no manual
validation code required.

### Field Constraint Reference

| Constraint | Meaning | Example |
|------------|---------|--------|
| `ge=X` | Greater than or equal to X | `ge=-90` for latitude |
| `le=X` | Less than or equal to X | `le=90` for latitude |
| `gt=X` | Strictly greater than X | `gt=0` for distance |
| `min_length=N` | String minimum length | `min_length=3` for device_id |
| `max_length=N` | String maximum length | `max_length=64` for device_id |

## Section 4: Illustrative Code — Pydantic Schema Pattern

Study the structure below. This shows the **pattern** for writing Pydantic schemas.
Notice how:
- `Field()` adds both validation rules AND documentation (shown in Swagger UI)
- `ConfigDict(from_attributes=True)` enables automatic ORM → Pydantic conversion
- Optional fields use Python's `| None` union type

**Do NOT copy this code directly.** Use it as a reference when implementing
the real schemas in `app/schemas/telemetry.py` and `app/schemas/device.py`.

In [ ]:
# ── ILLUSTRATIVE EXAMPLE ONLY — Do not run this cell ──
# This shows the PATTERN. Implement the real version in app/schemas/

from datetime import datetime
from typing import Optional
from pydantic import BaseModel, Field, ConfigDict


# Example: A request schema with strict field validation
class ExampleUploadSchema(BaseModel):
    """Illustrates how to structure a Pydantic request body."""

    device_id: str = Field(
        min_length=3,
        max_length=64,
        description="Unique identifier for this device",
        examples=["helmet-001-berkeley"],
    )

    # Latitude must be within the valid geographic range
    gps_lat: float = Field(
        ge=-90.0,   # ge = 'greater than or equal to'
        le=90.0,    # le = 'less than or equal to'
        description="Latitude in decimal degrees",
    )

    # Optional field — None if sensor is not present
    obstacle_distance_cm: Optional[int] = Field(
        default=None,
        gt=0,       # gt = 'greater than' (must be positive if provided)
        description="Distance to nearest obstacle in centimeters",
    )


# Example: A response schema with ORM mode enabled
class ExampleResponseSchema(BaseModel):
    """Illustrates how to convert a SQLAlchemy ORM object to a Pydantic response."""

    id: int
    device_id: str
    gps_lat: float
    created_at: datetime

    # This setting enables: ExampleResponseSchema.model_validate(orm_object)
    model_config = ConfigDict(from_attributes=True)


# ── How FastAPI uses these schemas automatically ──
#
# In a router, just declare the parameter type:
#
# @router.post("/example")
# async def example_endpoint(payload: ExampleUploadSchema):
#     # FastAPI validates the incoming JSON before this function runs
#     # If validation fails → 422 response returned automatically
#     print(payload.device_id)      # Guaranteed to be a str, 3-64 chars
#     print(payload.gps_lat)        # Guaranteed to be -90.0 ≤ value ≤ 90.0
#     ...

print("Study this pattern, then implement the real schemas in app/schemas/")

## Section 5: Illustrative Code — FastAPI Router Pattern

A FastAPI router groups related endpoints. Study how:
- `APIRouter()` creates a reusable router object
- `Depends(get_db)` injects a database session automatically
- The router function is **thin** — it calls a service and returns the result
- `status_code=201` sets the default success response code

In [ ]:
# ── ILLUSTRATIVE EXAMPLE ONLY — Do not run this cell ──
# This shows the PATTERN. Implement the real version in app/routers/

# from fastapi import APIRouter, Depends, HTTPException, status
# from sqlalchemy.orm import Session
# from app.database import get_db


# router = APIRouter()


# @router.post(
#     "/register",
#     response_model=DeviceResponse,      # Automatically convert return value → JSON
#     status_code=status.HTTP_201_CREATED, # Default success code
# )
# async def register_device(
#     payload: DeviceRegister,             # Pydantic validates this automatically
#     db: Session = Depends(get_db),       # FastAPI injects db session
# ):
#     """
#     Router is thin: call service, handle errors, return result.
#     No database code here!
#     """
#     try:
#         device = device_service.create_device(db, payload)  # service does the work
#         return device
#     except ValueError as exc:
#         raise HTTPException(
#             status_code=status.HTTP_409_CONFLICT,
#             detail=str(exc),
#         )


# ── App Factory Pattern ──
# In app/main.py, routers are registered with the app:
#
# app.include_router(
#     devices.router,      # The router object from app/routers/devices.py
#     prefix="/devices",   # All routes in this router get this URL prefix
#     tags=["Devices"],    # Groups endpoints in Swagger UI
# )

print("Study this pattern, then implement the real routers in app/routers/")

## Section 6: Testing with Swagger UI

FastAPI automatically generates an interactive API documentation page.
After you start the server, open:

**http://localhost:8000/docs**

You will see a page that lets you:
1. Click any endpoint (e.g., `POST /devices/register`)
2. Click **"Try it out"**
3. Fill in the request body with example JSON
4. Click **"Execute"**
5. See the response code and body

### Starting the development server

```bash
# Activate your virtual environment first
source .venv/bin/activate   # macOS/Linux
.venv\Scripts\activate      # Windows

# Start the server with auto-reload (restarts on code changes)
uvicorn app.main:app --reload
```

### Test Sequence for Lesson 1

Test your endpoints in this order (each step depends on the previous):

```
Step 1: GET  /health
        → Expected: 200 { "status": "ok" }

Step 2: POST /devices/register
        Body: { "device_id": "helmet-test-001", "owner_name": "Yixin Chen" }
        → Expected: 201 with device data including registered_at

Step 3: POST /devices/register (SAME body again)
        → Expected: 409 Conflict

Step 4: POST /telemetry/upload
        Body: valid payload with device_id = "helmet-test-001"
        → Expected: 201

Step 5: POST /telemetry/upload with gps_lat = 999.0
        → Expected: 422 Unprocessable Entity

Step 6: GET  /alerts/history
        → Expected: 200 with [] (empty list, no alerts triggered yet)
```

---

## 📌 YOUR TASK — File Map

Open each file listed below and implement every function marked `# TODO (Lesson 1)`.

Work in this order — each step depends on the previous:

### Step 1 — Environment Setup
```bash
# In the project root:
python -m venv .venv
.venv\Scripts\activate    # Windows
pip install -r requirements.txt
```

### Step 2 — Configuration (`app/config.py`)
- Add `DATABASE_URL`, `REDIS_URL`, `API_KEY` fields to `Settings`
- Implement `get_settings()` with `@lru_cache`
- Copy `.env.example` to `.env` (fill in your local DB credentials)

### Step 3 — Pydantic Schemas

| File | Class | What to implement |
|------|-------|-------------------|
| `app/schemas/device.py` | `DeviceRegister` | Fields: device_id (min 3 chars), owner_name, firmware_version (optional) |
| `app/schemas/device.py` | `DeviceResponse` | All Device fields + `ConfigDict(from_attributes=True)` |
| `app/schemas/telemetry.py` | `TelemetryUpload` | All fields with GPS constraints (ge/le), battery 0–100 |
| `app/schemas/telemetry.py` | `TelemetryResponse` | id, device_id, timestamp, alert_type |
| `app/schemas/alert.py` | `AlertResponse` | All Alert fields + `ConfigDict(from_attributes=True)` |

### Step 4 — Service Layer

| File | Function | What to implement |
|------|----------|-------------------|
| `app/services/device_service.py` | `create_device()` | Check duplicate, insert, return |
| `app/services/device_service.py` | `get_device_by_id()` | Query by device_id, return or None |
| `app/services/telemetry_service.py` | `create_telemetry_log()` | Insert TelemetryLog, return |
| `app/services/alert_service.py` | `create_alert()` | Map alert_type → severity, insert Alert |
| `app/services/alert_service.py` | `get_alerts()` | Query with optional device_id filter |

### Step 5 — Routers

| File | Endpoint | What to implement |
|------|----------|-------------------|
| `app/routers/devices.py` | `POST /devices/register` | Call service, catch ValueError → 409 |
| `app/routers/devices.py` | `GET /devices/{device_id}` | Call service, raise 404 if None |
| `app/routers/telemetry.py` | `POST /telemetry/upload` | Verify device exists, call service |
| `app/routers/telemetry.py` | `GET /telemetry/{id}/history` | Query with pagination |
| `app/routers/alerts.py` | `GET /alerts/history` | Call service, return list |

### Step 6 — App Factory (`app/main.py`)
- Import routers and register with `app.include_router()`
- Add `CORSMiddleware`
- Implement the `/health` endpoint

### Step 7 — Test in Swagger UI
```bash
uvicorn app.main:app --reload
# Open http://localhost:8000/docs
# Run the 6-step test sequence from Section 6 above
```

---

## ✅ Lesson 1 Self-Assessment Checklist

Check off each item when you have verified it works in Swagger UI:

- [ ] `GET /health` returns `200 { "status": "ok", "version": "0.1.0" }`
- [ ] `POST /devices/register` with valid payload returns `201` with `registered_at`
#- [ ] `POST /devices/register` with duplicate `device_id` returns `409`
- [ ] `POST /devices/register` with missing `owner_name` returns `422`
- [ ] `POST /devices/register` with `device_id = "ab"` (too short) returns `422`
#- [ ] `GET /devices/{device_id}` returns `200` for a registered device
#- [ ] `GET /devices/nonexistent-id` returns `404`
#- [ ] `POST /telemetry/upload` with valid payload returns `201`
#- [ ] `POST /telemetry/upload` with `gps_lat = 95.0` returns `422`
#- [ ] `POST /telemetry/upload` with unknown `device_id` returns `404`
#- [ ] `GET /alerts/history` returns `200` with `[]` when no alerts exist
- [ ] GitHub commit pushed with message: `feat(api): complete Lesson 1 API endpoints`

---

**When all boxes are checked → you are ready for Lesson 2.**